# Week 4 Coding Practice Solution: Can We Predict Penguin Body Mass?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/obscrivn/DataScience-book/blob/main/module04/week4_regression_practice_solution.ipynb)

This notebook provides one modeled solution to the student activity. Open-ended responses can differ when they use accurate evidence and responsible interpretation. Run the cells in order; the stored source intentionally contains no outputs.

## 1. Load, validate, and define the prediction problem

We use the unchanged 344-row course copy of Palmer Penguins. The response is `body_mass_g`; candidate predictors are measurements, species, and recorded sex. Missing values remain visible until a training-fitted preprocessing pipeline handles them.

**Source:** Horst AM, Hill AP, Gorman KB (2020). *palmerpenguins: Palmer Archipelago (Antarctica) penguin data*. Data originally collected by Kristen Gorman and the Palmer Station Long Term Ecological Research program. [Dataset documentation](https://allisonhorst.github.io/palmerpenguins/).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

sns.set_theme(style="whitegrid")
local_data = Path("../data/penguins.csv")
remote_data = (
    "https://raw.githubusercontent.com/obscrivn/"
    "DataScience-book/main/data/penguins.csv"
)
data_source = local_data if local_data.exists() else remote_data
penguins = pd.read_csv(data_source, na_values=["NA"])
expected_columns = {
    "species", "island", "bill_length_mm", "bill_depth_mm",
    "flipper_length_mm", "body_mass_g", "sex", "year"
}
assert penguins.shape == (344, 8)
assert set(penguins.columns) == expected_columns
print(f"Loaded {penguins.shape[0]} rows from {data_source}")
penguins.head()

In [ ]:
model_columns = [
    "body_mass_g", "flipper_length_mm", "bill_length_mm",
    "bill_depth_mm", "species", "sex"
]
display(penguins[model_columns].isna().sum().to_frame("missing"))
display(penguins[model_columns].describe(include="all").T)

### Possible response

The response is body mass in grams. Flipper length is a sensible first predictor because the Week 03 exploration suggests a strong positive association with body mass. Species may improve prediction by representing visible group structure, but its coefficient would still be an association rather than proof that species causes mass differences. These observations represent three penguin species sampled near Palmer Station during 2007-2009; performance should not be assumed to transfer unchanged to other species, locations, or time periods.

## 2. Explore before fitting

A reasonable prediction is a positive, roughly linear overall association with visible species clusters.

In [ ]:
explore_data = penguins.dropna(
    subset=["flipper_length_mm", "body_mass_g", "species"]
)
print(f"Scatterplot uses {len(explore_data)} of {len(penguins)} rows.")
fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(
    data=explore_data, x="flipper_length_mm", y="body_mass_g",
    hue="species", style="species", palette="colorblind",
    alpha=0.75, ax=ax,
)
ax.set(
    title="Body mass generally increases with flipper length",
    xlabel="Flipper length (mm)", ylabel="Body mass (g)",
)
ax.legend(title="Species", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

### Possible response

The points follow an overall upward trend without strong curvature, which supports trying a line. The separated species clusters warn that flipper length alone may miss group-related structure. The plot supports association, not the causal claim that making a flipper longer would make a penguin heavier.

## 3. Fit and interpret a simple regression

This full-data fit illustrates coefficients and residuals. It is not used as the final estimate of performance on unseen data.

In [ ]:
simple_data = penguins.dropna(
    subset=["flipper_length_mm", "body_mass_g"]
).copy()
simple_preview = LinearRegression()
simple_preview.fit(simple_data[["flipper_length_mm"]], simple_data["body_mass_g"])
simple_data["predicted_mass_g"] = simple_preview.predict(
    simple_data[["flipper_length_mm"]]
)
simple_data["residual_g"] = (
    simple_data["body_mass_g"] - simple_data["predicted_mass_g"]
)
print(f"Intercept: {simple_preview.intercept_:,.1f} g")
print(f"Slope: {simple_preview.coef_[0]:,.1f} g per 1 mm of flipper length")
print(f"Mean residual: {simple_data['residual_g'].mean():.6f} g")

### Possible response

In this dataset, penguins with flippers 1 mm longer are predicted by the simple model to have body mass about **49.7 grams higher**, on average. This is not a causal effect because other differences, including species, may explain part of the association. The intercept is where the fitted equation crosses zero flipper length; zero is far outside the observed range, so a negative predicted mass there has no realistic biological meaning.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.scatterplot(
    data=simple_data, x="flipper_length_mm", y="body_mass_g",
    color="#0072B2", alpha=0.65, ax=axes[0],
)
line_order = simple_data.sort_values("flipper_length_mm")
axes[0].plot(
    line_order["flipper_length_mm"], line_order["predicted_mass_g"],
    color="#D55E00", linewidth=2, label="Fitted line",
)
axes[0].set(
    title="Simple regression fit", xlabel="Flipper length (mm)",
    ylabel="Body mass (g)",
)
axes[0].legend()
sns.scatterplot(
    data=simple_data, x="predicted_mass_g", y="residual_g",
    color="#009E73", alpha=0.65, ax=axes[1],
)
axes[1].axhline(0, color="black", linewidth=1)
axes[1].set(
    title="Residuals from the simple model",
    xlabel="Predicted body mass (g)",
    ylabel="Residual: observed - predicted (g)",
)
plt.tight_layout()
plt.show()

### Possible response

Positive residuals are penguins heavier than predicted; negative residuals are lighter than predicted. The residuals show bands and uneven concentrations rather than completely patternless noise, consistent with the model omitting species structure. A near-zero mean residual only says positive and negative training errors balance; individual errors can still be hundreds of grams.

## 4. Split before learned preprocessing

We split raw rows first and stratify by species. Every learned preprocessing step will live inside a pipeline fitted on the training rows.

In [ ]:
predictor_columns = [
    "flipper_length_mm", "bill_length_mm", "bill_depth_mm",
    "species", "sex"
]
model_data = penguins.dropna(subset=["body_mass_g"]).copy()
X = model_data[predictor_columns]
y = model_data["body_mass_g"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=X["species"]
)
assert set(X_train.index).isdisjoint(X_test.index)
assert len(X_train) + len(X_test) == len(model_data)
print(f"Training rows: {len(X_train)}")
print(f"Test rows: {len(X_test)}")
display(
    pd.concat(
        [X_train["species"].value_counts(normalize=True).rename("train"),
         X_test["species"].value_counts(normalize=True).rename("test")],
        axis=1,
    ).round(3)
)

### Possible response: critique the leaky workflow

Filling and encoding before splitting lets the test rows influence the medians and the category vocabulary used to prepare training data. The response was not copied directly, but information about the future evaluation distribution still crossed the boundary. The resulting score can be optimistic because the workflow has already seen characteristics of the data it claims are unseen. The safe order is split first, then fit imputation and encoding on training data only.

## 5. Fit simple and multiple-predictor pipelines

The categorical encoder drops the first category, making Adelie the species reference and female the sex reference for this fitted training set.

In [ ]:
simple_features = ["flipper_length_mm"]
numeric_features = ["flipper_length_mm", "bill_length_mm", "bill_depth_mm"]
categorical_features = ["species", "sex"]
simple_preprocess = ColumnTransformer(
    [("numeric", SimpleImputer(strategy="median"), simple_features)],
    verbose_feature_names_out=False,
)
simple_model = Pipeline(
    [("preprocess", simple_preprocess), ("model", LinearRegression())]
)
categorical_preprocess = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(drop="first", handle_unknown="ignore")),
    ]
)
multiple_preprocess = ColumnTransformer(
    [
        ("numeric", SimpleImputer(strategy="median"), numeric_features),
        ("categorical", categorical_preprocess, categorical_features),
    ],
    verbose_feature_names_out=False,
)
multiple_model = Pipeline(
    [("preprocess", multiple_preprocess), ("model", LinearRegression())]
)
simple_model.fit(X_train, y_train)
multiple_model.fit(X_train, y_train)
print("Both pipelines were fit using training rows only.")

In [ ]:
feature_names = multiple_model.named_steps["preprocess"].get_feature_names_out()
coefficient_table = (
    pd.DataFrame(
        {
            "model_feature": feature_names,
            "coefficient_g": multiple_model.named_steps["model"].coef_,
        }
    )
    .assign(abs_coefficient_g=lambda frame: frame["coefficient_g"].abs())
    .sort_values("abs_coefficient_g", ascending=False)
)
coefficient_table.drop(columns="abs_coefficient_g").round(1)

### Possible response

Holding bill measurements, species, and recorded sex fixed, a 1 mm longer flipper is associated with about **18.0 g** higher predicted body mass. Holding the numeric predictors and species fixed, a penguin recorded as male is predicted to be about **337.0 g** heavier than the female reference group. These are conditional associations, not interventions. Ranking importance by coefficient magnitude would be misleading because the numeric features use different units and ranges, while categorical coefficients compare groups with reference categories.

## 6. Evaluate training and test performance

MAE and RMSE are reported in grams; RMSE penalizes large misses more strongly. R-squared summarizes variation accounted for relative to a mean prediction but does not give the typical error in grams.

In [ ]:
def regression_metrics(model, X_values, y_values):
    predictions = model.predict(X_values)
    return {
        "MAE_g": mean_absolute_error(y_values, predictions),
        "RMSE_g": np.sqrt(mean_squared_error(y_values, predictions)),
        "R_squared": r2_score(y_values, predictions),
    }

rows = []
for model_name, model in {
    "Simple: flipper length": simple_model,
    "Multiple: measurements + categories": multiple_model,
}.items():
    for split_name, X_values, y_values in [
        ("train", X_train, y_train),
        ("test", X_test, y_test),
    ]:
        rows.append(
            {"model": model_name, "split": split_name,
             **regression_metrics(model, X_values, y_values)}
        )
performance = pd.DataFrame(rows)
assert np.isfinite(performance[["MAE_g", "RMSE_g", "R_squared"]]).all().all()
performance.round({"MAE_g": 1, "RMSE_g": 1, "R_squared": 3})

### Possible response

The multiple model improves test MAE from **299.7 g to 239.4 g** and test RMSE from **373.4 g to 288.8 g**. The reductions are about 60 g and 85 g, respectively. Whether that matters depends on the decision: it could be adequate for a low-stakes educational estimate but not automatically for wildlife health intervention. Test performance is slightly better than training performance for both models, so this split does not show the usual train-good/test-poor signature of overfitting. A single split still cannot prove that overfitting or instability is impossible.

In [ ]:
test_predictions = multiple_model.predict(X_test)
test_results = pd.DataFrame(
    {"observed_mass_g": y_test, "predicted_mass_g": test_predictions}
)
test_results["residual_g"] = (
    test_results["observed_mass_g"] - test_results["predicted_mass_g"]
)
low = min(test_results["observed_mass_g"].min(), test_results["predicted_mass_g"].min())
high = max(test_results["observed_mass_g"].max(), test_results["predicted_mass_g"].max())
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].scatter(
    test_results["observed_mass_g"], test_results["predicted_mass_g"],
    color="#0072B2", alpha=0.75,
)
axes[0].plot([low, high], [low, high], linestyle="--", color="black")
axes[0].set(
    title="Multiple model predictions on unseen test rows",
    xlabel="Observed body mass (g)", ylabel="Predicted body mass (g)",
)
axes[1].scatter(
    test_results["predicted_mass_g"], test_results["residual_g"],
    color="#009E73", alpha=0.75,
)
axes[1].axhline(0, color="black", linewidth=1)
axes[1].set(
    title="Test residuals", xlabel="Predicted body mass (g)",
    ylabel="Residual: observed - predicted (g)",
)
plt.tight_layout()
plt.show()
test_results.reindex(
    test_results["residual_g"].abs().sort_values(ascending=False).index
).head()

## 7. Generate and communicate new predictions

The example values stay near ranges represented in the training data.

In [ ]:
new_penguins = pd.DataFrame(
    [
        {"flipper_length_mm": 190, "bill_length_mm": 39,
         "bill_depth_mm": 18, "species": "Adelie", "sex": "female"},
        {"flipper_length_mm": 220, "bill_length_mm": 49,
         "bill_depth_mm": 15, "species": "Gentoo", "sex": "male"},
    ]
)
new_penguins.assign(
    predicted_body_mass_g=multiple_model.predict(new_penguins).round(0)
)

### Possible response

The model predicts about **3,504 g** for the hypothetical Adelie row. This is an estimate based on penguins similar to those sampled, and the test MAE shows that individual predictions commonly miss by hundreds of grams. A species coefficient does not describe the effect of changing species: species cannot be experimentally switched, and it is associated with many biological and sampling differences not isolated by this observational model.

## 8. Independent transfer solution: remove `sex`

This comparison holds the original train/test rows fixed and changes one modeling decision: whether recorded sex is included. That isolates the predictor decision more cleanly than changing the random split at the same time.

In [ ]:
no_sex_categorical = ["species"]
no_sex_preprocess = ColumnTransformer(
    [
        ("numeric", SimpleImputer(strategy="median"), numeric_features),
        ("categorical", categorical_preprocess, no_sex_categorical),
    ],
    verbose_feature_names_out=False,
)
no_sex_model = Pipeline(
    [("preprocess", no_sex_preprocess), ("model", LinearRegression())]
)
no_sex_model.fit(X_train, y_train)
no_sex_test = {
    "model": "Multiple without sex",
    "split": "test",
    **regression_metrics(no_sex_model, X_test, y_test),
}
comparison = pd.concat(
    [performance.query("split == 'test'"), pd.DataFrame([no_sex_test])],
    ignore_index=True,
)
comparison.round({"MAE_g": 1, "RMSE_g": 1, "R_squared": 3})

### Possible response

I removed recorded sex to test whether a simpler model could retain similar unseen-data performance. On the same test rows, MAE changes only from **239.4 g to 240.5 g**, while RMSE increases from **288.8 g to 297.6 g** and R-squared falls from **0.872 to 0.865**. The small MAE difference suggests the simpler model may be reasonable when sex is unavailable, although its larger RMSE indicates somewhat worse large errors. This single dataset and split still do not establish performance for new populations or future collection conditions.

## 9. Final workflow and claim audit

A defensible analysis checks that the response and predictors match the question, splits raw rows before learned preprocessing, evaluates unseen data in meaningful units, examines residuals, avoids extrapolation, and separates prediction from causation.

### Possible exit reflection

Leakage can be especially easy for generated code to hide because a preprocessing call may appear far above the train/test split or inside a helper function. Before trusting the result, I would request the exact order of operations, evidence that each imputer and encoder was fit only on training rows, and training/test metrics produced from a reproducible split.